# 04 · TOP 20 MOMENTUM · RECONCILIATION (PAPER / LIVE vs ENGINE)
Reconciles what you actually did (screener + broker) with `t20_engine.py`, one layer at a time. **Run it only after notebook 03 is validated**, with >= 6 rebalances of paper trading (TIS minimum).

| Layer | Your file | What is compared with the engine |
| :-- | :-- | :-- |
| 1. Signals | `signals_log.csv` (`date,ticker,rank,perf_quarter`) | Top N of the screener (Finviz) vs the engine Top N on the same rebalance date |
| 2. Trades | `trades_log.csv` (`datetime,ticker,side,shares,price`) | Entries / exits: dates, prices (realised slippage), missing and extra trades |
| 3. Equity | `equity_log.csv` (`date,equity`) | Daily return of your account vs the engine over the common window |

Place the files in `runs/<variant>/live/`. Finviz "Perf Quarter" is ~3 calendar months, the engine uses 63 trading days: a **partial** overlap is expected, so the gate uses tolerances, not equality.

In [ ]:
from t20_tools import *
VARIANT = "t20_sp500_v1"
S = setup(VARIANT)
eng, P, rules, panel, reg = S.eng, S.P, S.rules, S.panel, S.reg
check, N = reg.check, S.P["top_n"]; LIVE = S.RUN / "live"; LIVE.mkdir(exist_ok=True)
TOL = dict(overlap=0.70, entry_days=1, slip_bps=15.0, missing=0.10, te_ann=0.03)
files = {k: LIVE / f"{k}_log.csv" for k in ("signals", "trades", "equity")}; print({k: v.exists() for k, v in files.items()})
sig = pd.read_csv(files["signals"], parse_dates=["date"]) if files["signals"].exists() else None
trd = pd.read_csv(files["trades"], parse_dates=["datetime"]) if files["trades"].exists() else None
eqx = pd.read_csv(files["equity"], parse_dates=["date"], index_col="date")["equity"] if files["equity"].exists() else None
start = min(x for x in (sig["date"].min() if sig is not None else None, trd["datetime"].min() if trd is not None else None) if x is not None) if (sig is not None or trd is not None) else None
assert start is not None, f"Put at least one log in {LIVE} (see the table above)."

## 1. ENGINE REFERENCE (SAME WINDOW, FROM THE FIRST LOGGED DAY)

In [ ]:
ref = eng.run_backtest(panel, **rules, sim_start=start - pd.Timedelta(days=1)); rt = ref["trades"]
print(f"Engine from {ref['ret'].index[0]:%Y-%m-%d}: {len(rt)} trades. NOTE: the engine starts flat at the first day; positions you already held before it will show up as 'extra' in your log.")

## 2. SIGNAL RECONCILIATION (TOP N PER REBALANCE)

In [ ]:
reb, _ = eng.calendar_flags(panel.dates, rules["rebalance"]); mom = panel.mom(rules["lookback_days"]); sS, sF = panel.sma(rules["stock_ma"]), panel.sma(rules["fast_ma"])
def engine_top(d):
    t = panel.dates.searchsorted(d)
    el = panel.member[t] & ~np.isnan(mom[t]) & (panel.adv[t] >= rules["min_adv_shares"]) & (panel.close[t] > sS[t]) & (panel.close[t] > sF[t]); ix = np.flatnonzero(el)
    return [panel.tickers[j] for j in ix[np.argsort(-mom[t, ix])][:N]]
if sig is not None:
    rows = []
    for d, g in sig.groupby("date"):
        a, b = set(g.sort_values("rank")["ticker"].str.upper().head(N)), set(engine_top(d)); rows.append(dict(date=d, overlap=len(a & b) / N, only_log=sorted(a - b), only_engine=sorted(b - a)))
    sg = pd.DataFrame(rows); print(sg.to_string(index=False))
    check("2. Signals", f"Mean Top {N} overlap with the engine >= {TOL['overlap']:.0%}", sg["overlap"].mean() >= TOL["overlap"], f"{sg['overlap'].mean():.0%} over {len(sg)} rebalances")
    check("2. Signals", "No rebalance below 50% overlap", (sg["overlap"] >= .5).all(), f"min {sg['overlap'].min():.0%}")
else: check("2. Signals", "signals_log.csv present", None, "skipped")

## 3. TRADE RECONCILIATION

In [ ]:
if trd is not None:
    trd["ticker"] = trd["ticker"].str.upper(); buys = trd[trd["side"].str.lower() == "buy"]; sells = trd[trd["side"].str.lower() == "sell"]
    def match(log, col, eng_col, side):
        rows = []
        for r in log.itertuples():
            c = rt[(rt["ticker"] == r.ticker)]; 
            if c.empty: rows.append(dict(ticker=r.ticker, date=r.datetime, matched=False)); continue
            k = (c[eng_col] - r.datetime.normalize()).abs().idxmin(); e = c.loc[k]; dd = abs((e[eng_col] - r.datetime.normalize()).days)
            px_e = e["entry_price"] if side == "buy" else e["exit_price"]
            rows.append(dict(ticker=r.ticker, date=r.datetime, matched=dd <= 4, day_diff=dd, px_log=r.price, px_engine=px_e, slip_bps=(r.price / px_e - 1) * 1e4 * (1 if side == "buy" else -1)))
        return pd.DataFrame(rows)
    mb, ms = match(buys, "datetime", "entry_time", "buy"), match(sells, "datetime", "exit_time", "sell")
    for nm, m in (("BUYS", mb), ("SELLS", ms)):
        print(nm, f"| matched {m['matched'].mean():.0%} | mean day diff {m.loc[m['matched'], 'day_diff'].mean():.1f} | mean slip vs engine {m.loc[m['matched'], 'slip_bps'].mean():+.1f} bps (engine assumes {P['slip_bps_per_side']:g} per side on its own price)")
    check("3. Trades", f"Logged buys that the engine also made >= {1 - TOL['missing']:.0%}", mb["matched"].mean() >= 1 - TOL["missing"], f"{mb['matched'].mean():.0%}")
    check("3. Trades", f"Logged sells that the engine also made >= {1 - TOL['missing']:.0%}", ms["matched"].mean() >= 1 - TOL["missing"], f"{ms['matched'].mean():.0%}")
    check("3. Trades", f"Entries within {TOL['entry_days']} calendar day(s) of the engine", (mb.loc[mb["matched"], "day_diff"] <= TOL["entry_days"] + 2).mean() >= .9, f"{(mb.loc[mb['matched'], 'day_diff'] <= TOL['entry_days'] + 2).mean():.0%}")
    check("3. Trades", f"Realised slippage vs engine price <= {TOL['slip_bps']:g} bps", abs(pd.concat([mb, ms])["slip_bps"].mean()) <= TOL["slip_bps"], f"{pd.concat([mb, ms])['slip_bps'].mean():+.1f} bps")
    miss_eng = rt[(rt["entry_time"] >= trd["datetime"].min().normalize()) & ~rt["ticker"].isin(trd["ticker"])]
    print(f"\nEngine trades never done in your log: {len(miss_eng)}"); print(miss_eng[["ticker", "entry_time", "exit_time", "reason"]].head(20).to_string(index=False))
else: check("3. Trades", "trades_log.csv present", None, "skipped")

## 4. EQUITY RECONCILIATION

In [ ]:
if eqx is not None:
    r_live = eqx.pct_change().dropna(); r_eng = ref["ret"].reindex(r_live.index).dropna(); r_live = r_live.reindex(r_eng.index); te = (r_live - r_eng).std() * np.sqrt(252)
    print(f"Common days {len(r_eng)} | live {100 * ((1 + r_live).prod() - 1):+.2f}% vs engine {100 * ((1 + r_eng).prod() - 1):+.2f}% | tracking error {100 * te:.1f}% annualised | corr {r_live.corr(r_eng):.2f}")
    check("4. Equity", f"Tracking error <= {TOL['te_ann']:.0%} annualised", te <= TOL["te_ann"], f"{100 * te:.1f}%"); check("4. Equity", "Daily correlation >= 0.95", r_live.corr(r_eng) >= .95, f"{r_live.corr(r_eng):.2f}")
else: check("4. Equity", "equity_log.csv present", None, "skipped")

## 5. VERDICT

In [ ]:
ESSENTIAL = [("2. Signals", "Mean Top", "Your screener and the engine pick different stocks: align the Finviz filters (footer must show the SMA50 / SMA200 filters) or document the exception."),
    ("3. Trades", "Logged buys", "Trades you made that the engine does not: review execution errors."), ("3. Trades", "Logged sells", "Exits not explained by the rules: review."),
    ("3. Trades", "Realised slippage", "Real costs are above the model: re-run notebook 02 section 3.5 with the real slippage."), ("4. Equity", "Tracking error", "Account diverges from the engine.")]
go = gate(reg, ESSENTIAL); S.vc.save_json(S.cfg, "04_reconciliation.json", dict(variant=S.cfg["id"], reconciled=bool(go), checks=reg.df().to_dict("records")))